# R3 — Raw Sequential Sector-Price Representation Ablation

Input-representation ablation only: universe, R1 primary target, model definitions (LR / NN v1 `SmallMLP` / NN v2 `WideSingleLayerMLP` ensemble), and the R1 walk-forward fold schedule (test years 2008-2026) are all held fixed. Only the input changes: raw sequential sector prices (+ target-sector one-hot) instead of R1's engineered features. See `docs/weekly_sector_rotation_r3_raw_sequence_test.md` for the full write-up; this notebook is the executable source of the CSVs it cites. Does not implement R4 (MIMO).

In [1]:
"""
R3: raw sequential sector-price representation vs. R1's engineered-feature representation,
for the R1 primary target (target_abs_1pct_next_week). Input-representation ablation only --
same LR / NN v1 / NN v2 model definitions and walk-forward schedule as R1, no new architectures.
"""
import json
import time
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

t0 = time.time()

REPO_ROOT = Path("/Users/gavinhussey/Downloads/AtlasQuant")
PRICES_ROOT = REPO_ROOT / "data" / "raw" / "weekly_sector_rotation" / "prices"
FEATURE_PANEL_PATH = REPO_ROOT / "data" / "processed" / "weekly_sector_rotation" / "feature_panel.csv"
OUTPUTS_ROOT = REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation" / "outputs"
R1_TARGETS_PATH = OUTPUTS_ROOT / "r1_target_definitions_full_panel.csv"

SECTOR_ETFS = ["VGT", "VHT", "VCR", "VOX", "VFH", "VIS", "VDC", "VPU", "VAW", "VNQ", "VDE"]
BENCHMARK_ETF = "SPY"

N_VALUES = [4, 8, 13, 26, 52]
RANK_K_VALUES = [2, 3]
MIN_TRAIN_YEARS = 3
BLOCK_SIZE = 8
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 20260812
SHUFFLE_SEED = 20260812

# ---------------------------------------------------------------------------
# 1. Load the R1-validated targets (exact reuse, not recomputed) + the panel
#    (for FEATURE_COLUMNS, needed only to regenerate the R1 engineered baseline's
#    OOF predictions for the paired comparison -- Step 9's explicitly permitted use).
# ---------------------------------------------------------------------------
date_cols_panel = ["date", "week_end", "actual_last_trading_date", "feature_cutoff_timestamp",
                    "next_week_first_trading_date", "next_week_last_trading_date"]
panel = pd.read_csv(FEATURE_PANEL_PATH, parse_dates=date_cols_panel)
panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)
AUDIT_COLUMNS = ["week_end", "actual_last_trading_date", "week_complete", "feature_cutoff_timestamp",
                  "next_week_first_trading_date", "next_week_last_trading_date"]
FEATURE_COLUMNS = [c for c in panel.columns if c not in {"date", "symbol", *AUDIT_COLUMNS}]

r1_targets = pd.read_csv(R1_TARGETS_PATH, parse_dates=["date"])
panel = panel.merge(
    r1_targets[["date", "symbol", "target_abs_1pct_next_week", "next_week_open_to_close_return",
                "next_week_spy_return", "next_week_excess_return"]],
    on=["date", "symbol"], how="left",
)
panel["year"] = panel["date"].dt.year

PANEL_DATES = sorted(panel["date"].unique())
DATE_TO_IDX = {d: i for i, d in enumerate(PANEL_DATES)}
print(f"[{time.time()-t0:.0f}s] loaded panel, {len(PANEL_DATES)} weeks, "
      f"{PANEL_DATES[0]} -> {PANEL_DATES[-1]}")

# ---------------------------------------------------------------------------
# 2. Build the validated weekly sector adjusted-close matrix (11 columns, no SPY),
#    reindexed onto the panel's own already-validated date list -- same resample
#    convention as feature_selection.ipynb (W-FRI, last observed close that week).
# ---------------------------------------------------------------------------
def resample_weekly_close(daily_df: pd.DataFrame) -> pd.Series:
    return daily_df["adjusted_close"].resample("W-FRI").last()

sector_close = {}
for s in SECTOR_ETFS:
    df = pd.read_csv(PRICES_ROOT / f"{s}.csv", parse_dates=["date"]).set_index("date").sort_index()
    sector_close[s] = resample_weekly_close(df)
sector_close_matrix = pd.DataFrame(sector_close)[SECTOR_ETFS].reindex(PANEL_DATES)

n_nan_cells = int(sector_close_matrix.isna().sum().sum())
print(f"[{time.time()-t0:.0f}s] sector weekly adjusted-close matrix built, shape={sector_close_matrix.shape}, "
      f"NaN cells={n_nan_cells}")
assert n_nan_cells == 0, "unexpected gap in the weekly sector price matrix -- investigate before proceeding"
SECTOR_MATRIX_VALUES = sector_close_matrix.to_numpy()  # shape (n_weeks, 11), chronological order

[0s] loaded panel, 1141 weeks, 2004-10-01 00:00:00 -> 2026-08-07 00:00:00
[0s] sector weekly adjusted-close matrix built, shape=(1141, 11), NaN cells=0


## 3. STEP 3/8 -- N-sample audit (before building any model input).

In [2]:
n_weeks_total = len(PANEL_DATES)
# R1's fold schedule was derived from panel rows with COMPLETE engineered features
# (dropna on FEATURE_COLUMNS, which need up to a 26-week warmup) -- that pushes R1's
# usable years to start at 2005, giving test_years = 2008-2026 (19 folds). The raw
# panel itself starts Oct 2004, so deriving years directly from `panel` (as an earlier
# version of this script mistakenly did) spuriously adds a test_year=2007 fold with a
# too-small training pool that R1 never had. Reproduce R1's exact year list instead.
_md_for_years = panel.dropna(subset=FEATURE_COLUMNS + ["target_abs_1pct_next_week"])
years_all = sorted(_md_for_years["year"].unique())
test_years = years_all[MIN_TRAIN_YEARS:]  # identical fixed R1 fold schedule, 2008-2026
assert test_years == list(range(2008, 2027)), f"fold schedule does not match R1's validated 2008-2026 schedule: {test_years}"

n_audit_rows = []
for N in N_VALUES:
    first_usable_idx = N - 1
    first_usable_date = PANEL_DATES[first_usable_idx]
    n_lost_warmup = first_usable_idx
    n_usable_weeks = n_weeks_total - n_lost_warmup
    per_fold = []
    for ty in test_years:
        train_idxs = [i for i, d in enumerate(PANEL_DATES) if pd.Timestamp(d).year < ty and i >= first_usable_idx]
        test_idxs = [i for i, d in enumerate(PANEL_DATES) if pd.Timestamp(d).year == ty and i >= first_usable_idx]
        per_fold.append({
            "N": N, "test_year": ty, "n_train_weeks": len(train_idxs), "n_test_weeks": len(test_idxs),
            "n_train_samples": len(train_idxs) * 11, "n_test_samples": len(test_idxs) * 11,
            "fold_valid": len(test_idxs) > 0 and len(train_idxs) > 0,
        })
    n_audit_rows.extend(per_fold)
    invalid_folds = [f for f in per_fold if not f["fold_valid"]]
    print(f"[{time.time()-t0:.0f}s] N={N}: first usable week={first_usable_date.date()}, "
          f"{n_lost_warmup} weeks lost to warmup, {n_usable_weeks} usable weeks, "
          f"{len(invalid_folds)} invalid folds")

n_sample_audit = pd.DataFrame(n_audit_rows)
n_sample_audit.to_csv(OUTPUTS_ROOT / "r3_n_sample_audit.csv", index=False)

# ---------------------------------------------------------------------------
# 4. STEP 14 -- VDE data-gap impact on the raw sequence (independently re-verified,
#    not assumed): the weekly resample absorbs the missing 2026-07-31 daily bar via
#    "last observed price that week" (confirmed zero NaNs above). The gap's only
#    effect remains what R1 already found: 1 row's *target* (VDE, week 2026-07-24)
#    is undefined because the raw open/close execution-return lookup for that
#    specific forward week failed. That row is excluded from every N here via the
#    shared target-NaN dropna, identically to R1 -- no different handling needed.
# ---------------------------------------------------------------------------
vde_gap_date = pd.Timestamp("2026-07-24")
gap_rows = []
for N in N_VALUES:
    # Does the gap week (2026-07-31, VDE's actual missing trading day) fall inside
    # any sample's N-week input window? It's a valid (non-NaN) data point in the
    # weekly matrix regardless (last-available-price convention), so no sequence
    # sample is invalidated by it for any N -- verified, not assumed.
    affected_target_rows = int(((panel["symbol"] == "VDE") & (panel["date"] == vde_gap_date) &
                                 panel["target_abs_1pct_next_week"].isna()).sum())
    gap_rows.append({
        "N": N, "sequence_samples_invalidated": 0,
        "target_rows_invalidated": affected_target_rows,
        "affected_symbol": "VDE", "affected_week": str(vde_gap_date.date()),
        "note": "weekly resample uses last-observed price that week (2026-07-30 close); "
                "the raw daily gap on 2026-07-31 never enters any backward-looking sequence "
                "as a NaN. Only the forward-looking target for this one (date, symbol) row, "
                "already excluded in R1, is affected.",
    })
r3_data_gap_impact = pd.DataFrame(gap_rows)
r3_data_gap_impact.to_csv(OUTPUTS_ROOT / "r3_data_gap_impact.csv", index=False)
print(f"[{time.time()-t0:.0f}s] Step 14 (VDE gap impact) done -- 0 sequence samples invalidated, "
      f"1 target row invalidated (matches R1) for every N.")

# ---------------------------------------------------------------------------
# 5. Build raw-sequence sample tables per N: one row per (date, target_sector),
#    X = flatten(N-week x 11-sector price window) + one-hot(target sector).
# ---------------------------------------------------------------------------
target_lookup = panel.set_index(["date", "symbol"])[
    ["target_abs_1pct_next_week", "next_week_open_to_close_return", "next_week_spy_return", "year"]
]

def build_sequence_samples(N: int):
    first_usable_idx = N - 1
    rows_meta = []
    X_seq_list = []
    onehot_list = []
    for i in range(first_usable_idx, n_weeks_total):
        window = SECTOR_MATRIX_VALUES[i - N + 1: i + 1, :]  # (N, 11), chronological
        flat = window.flatten()  # row-major: week-by-week, each week's 11 sectors in SECTOR_ETFS order
        date = PANEL_DATES[i]
        for j, sym in enumerate(SECTOR_ETFS):
            key = (date, sym)
            if key not in target_lookup.index:
                continue
            row = target_lookup.loc[key]
            if pd.isna(row["target_abs_1pct_next_week"]):
                continue
            onehot = np.zeros(11, dtype=np.float64)
            onehot[j] = 1.0
            X_seq_list.append(flat)
            onehot_list.append(onehot)
            rows_meta.append({
                "date": date, "symbol": sym, "year": int(row["year"]),
                "label_binary": int(row["target_abs_1pct_next_week"]),
                "next_week_open_to_close_return": row["next_week_open_to_close_return"],
                "next_week_spy_return": row["next_week_spy_return"],
            })
    meta = pd.DataFrame(rows_meta)
    X_seq = np.vstack(X_seq_list) if X_seq_list else np.empty((0, N * 11))
    X_onehot = np.vstack(onehot_list) if onehot_list else np.empty((0, 11))
    return meta, X_seq, X_onehot

samples_by_N = {}
for N in N_VALUES:
    meta, X_seq, X_onehot = build_sequence_samples(N)
    samples_by_N[N] = {"meta": meta, "X_seq": X_seq, "X_onehot": X_onehot}
    print(f"[{time.time()-t0:.0f}s] N={N}: {len(meta):,} samples built "
          f"(input dim = {N*11} raw + 11 one-hot = {N*11+11})")

print(f"[{time.time()-t0:.0f}s] Step 2/5 (raw sequence construction) done.")

[0s] N=4: first usable week=2004-10-22, 3 weeks lost to warmup, 1138 usable weeks, 0 invalid folds
[0s] N=8: first usable week=2004-11-19, 7 weeks lost to warmup, 1134 usable weeks, 0 invalid folds
[0s] N=13: first usable week=2004-12-24, 12 weeks lost to warmup, 1129 usable weeks, 0 invalid folds
[0s] N=26: first usable week=2005-03-25, 25 weeks lost to warmup, 1116 usable weeks, 0 invalid folds
[0s] N=52: first usable week=2005-09-23, 51 weeks lost to warmup, 1090 usable weeks, 0 invalid folds
[0s] Step 14 (VDE gap impact) done -- 0 sequence samples invalidated, 1 target row invalidated (matches R1) for every N.


[1s] N=4: 12,506 samples built (input dim = 44 raw + 11 one-hot = 55)
[1s] N=8: 12,462 samples built (input dim = 88 raw + 11 one-hot = 99)


[2s] N=13: 12,407 samples built (input dim = 143 raw + 11 one-hot = 154)


[2s] N=26: 12,264 samples built (input dim = 286 raw + 11 one-hot = 297)


[3s] N=52: 11,978 samples built (input dim = 572 raw + 11 one-hot = 583)
[3s] Step 2/5 (raw sequence construction) done.


## 6. Model definitions -- verbatim from R1 (only input dimensionality varies).

In [3]:
NN_SEED = 20260812
HIDDEN_1, HIDDEN_2 = 16, 8
DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 1e-3
MAX_EPOCHS = 300
PATIENCE = 15


class SmallMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, HIDDEN_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_1, HIDDEN_2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_fold_mlp(X_fit, y_fit, X_val, y_val, n_features, seed=NN_SEED):
    torch.manual_seed(seed)
    model = SmallMLP(n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32)
    y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0
    for _ in range(MAX_EPOCHS):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_fit_t), y_fit_t)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model


V2_HIDDEN = 32
V2_DROPOUT = 0.4
V2_WEIGHT_DECAY = 1e-3
V2_LEARNING_RATE = 1e-3
V2_MAX_EPOCHS = 300
V2_PATIENCE = 15
N_SEEDS = 5
SEED_BASE = 20260812


class WideSingleLayerMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, V2_HIDDEN), nn.ReLU(), nn.Dropout(V2_DROPOUT),
            nn.Linear(V2_HIDDEN, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_fold_mlp_v2(X_fit, y_fit, X_val, y_val, n_features, seed):
    torch.manual_seed(seed)
    model = WideSingleLayerMLP(n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=V2_LEARNING_RATE, weight_decay=V2_WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32)
    y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0
    for _ in range(V2_MAX_EPOCHS):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_fit_t), y_fit_t)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= V2_PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model


# ---------------------------------------------------------------------------
# 7. Walk-forward evaluators for the raw-sequence representation. Same expanding
#    year-based schedule, same fit/val split rule, same scaler-fit boundary as R1
#    -- only the input matrix (raw sequence + one-hot, fold-fit StandardScaler on
#    the continuous block only) differs from R1's engineered-feature matrix.
# ---------------------------------------------------------------------------
def make_fold_matrix(X_seq, X_onehot, idx, scaler=None, fit=False):
    seq = X_seq[idx]
    if fit:
        scaler = StandardScaler().fit(seq)
    seq_scaled = scaler.transform(seq)
    return np.hstack([seq_scaled, X_onehot[idx]]), scaler


def walk_forward_raw_lr(meta, X_seq, X_onehot, years_, test_years_):
    oof_frames = []
    fold_rows = []
    for test_year in test_years_:
        train_mask = (meta["year"] < test_year).to_numpy()
        test_mask = (meta["year"] == test_year).to_numpy()
        if train_mask.sum() == 0 or test_mask.sum() == 0:
            continue
        X_train, scaler = make_fold_matrix(X_seq, X_onehot, train_mask, fit=True)
        X_test, _ = make_fold_matrix(X_seq, X_onehot, test_mask, scaler=scaler)
        y_train = meta.loc[train_mask, "label_binary"].to_numpy()
        y_test = meta.loc[test_mask, "label_binary"].to_numpy()
        clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
        predicted_proba = clf.predict_proba(X_test)[:, 1]

        fold = meta.loc[test_mask, ["date", "symbol", "next_week_open_to_close_return",
                                     "next_week_spy_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)
        fold_auc = roc_auc_score(y_test, predicted_proba) if len(np.unique(y_test)) > 1 else np.nan
        fold_rows.append(_fold_metrics_row(test_year, train_mask, test_mask, y_train, y_test, predicted_proba))
    oof = pd.concat(oof_frames, ignore_index=True)
    return oof, pd.DataFrame(fold_rows)


def walk_forward_raw_nn(meta, X_seq, X_onehot, years_, test_years_, n_features, v2=False):
    oof_frames = []
    fold_rows = []
    for test_year in test_years_:
        train_years_available = [y for y in years_ if y < test_year]
        if len(train_years_available) < 2:
            continue
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]
        train_mask = meta["year"].isin(train_years_available).to_numpy()
        fit_mask = meta["year"].isin(fit_years).to_numpy()
        val_mask = (meta["year"] == val_year).to_numpy()
        test_mask = (meta["year"] == test_year).to_numpy()
        if train_mask.sum() == 0 or test_mask.sum() == 0 or fit_mask.sum() == 0 or val_mask.sum() == 0:
            continue

        _, scaler = make_fold_matrix(X_seq, X_onehot, train_mask, fit=True)
        X_fit, _ = make_fold_matrix(X_seq, X_onehot, fit_mask, scaler=scaler)
        X_val, _ = make_fold_matrix(X_seq, X_onehot, val_mask, scaler=scaler)
        X_test, _ = make_fold_matrix(X_seq, X_onehot, test_mask, scaler=scaler)
        y_fit = meta.loc[fit_mask, "label_binary"].to_numpy()
        y_val = meta.loc[val_mask, "label_binary"].to_numpy()
        y_test = meta.loc[test_mask, "label_binary"].to_numpy()

        if v2:
            seed_predictions = []
            for seed_offset in range(N_SEEDS):
                model = train_fold_mlp_v2(X_fit, y_fit, X_val, y_val, n_features, SEED_BASE + seed_offset)
                with torch.no_grad():
                    seed_predictions.append(torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy())
            predicted_proba = np.mean(seed_predictions, axis=0)
        else:
            model = train_fold_mlp(X_fit, y_fit, X_val, y_val, n_features)
            with torch.no_grad():
                predicted_proba = torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy()

        fold = meta.loc[test_mask, ["date", "symbol", "next_week_open_to_close_return",
                                     "next_week_spy_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)
        fold_rows.append(_fold_metrics_row(test_year, train_mask, test_mask, y_fit, y_test, predicted_proba))
    oof = pd.concat(oof_frames, ignore_index=True)
    return oof, pd.DataFrame(fold_rows)


def _fold_metrics_row(test_year, train_mask, test_mask, y_train, y_test, predicted_proba):
    fold_auc = roc_auc_score(y_test, predicted_proba) if len(np.unique(y_test)) > 1 else np.nan
    fold_prauc = average_precision_score(y_test, predicted_proba) if len(np.unique(y_test)) > 1 else np.nan
    fold_brier = brier_score_loss(y_test, predicted_proba)
    return {
        "test_year": test_year, "n_train": int(np.sum(train_mask)), "n_test": int(np.sum(test_mask)),
        "train_positive_rate": float(np.mean(y_train)), "test_positive_rate": float(np.mean(y_test)),
        "fold_auc": fold_auc, "fold_pr_auc": fold_prauc, "fold_brier": fold_brier,
    }

## 8. Top-K metrics (reused definition from R1) + weekly series for bootstrapping.

In [4]:
def topk_metrics_and_series(oof: pd.DataFrame, base_rate: float, k: int):
    weekly_precision, weekly_any_pos, weekly_all_pos = [], [], []
    weekly_mean_ret, weekly_all_sector_mean, weekly_spy_ret, weekly_bottomk_mean = [], [], [], []
    dates_used = []
    for date, g in oof.groupby("date"):
        if len(g) < 11:
            continue  # need the full 11-sector cross-section that week
        ranked = g["predicted_proba"].rank(ascending=False, method="first")
        n = len(g)
        top = g.loc[ranked <= k]
        bottom = g.loc[ranked > n - k]
        if top["next_week_open_to_close_return"].isna().any() or top["label_binary"].isna().any():
            continue
        weekly_precision.append(top["label_binary"].mean())
        weekly_any_pos.append(float((top["label_binary"] == 1).any()))
        weekly_all_pos.append(float((top["label_binary"] == 1).all()))
        weekly_mean_ret.append(top["next_week_open_to_close_return"].mean())
        weekly_all_sector_mean.append(g["next_week_open_to_close_return"].mean())
        weekly_spy_ret.append(g["next_week_spy_return"].iloc[0])
        weekly_bottomk_mean.append(bottom["next_week_open_to_close_return"].mean())
        dates_used.append(date)

    series = {
        "date": dates_used,
        "precision": pd.Series(weekly_precision), "selected_ret": pd.Series(weekly_mean_ret),
        "all_sector_mean": pd.Series(weekly_all_sector_mean), "spy_ret": pd.Series(weekly_spy_ret),
        "bottomk_mean": pd.Series(weekly_bottomk_mean),
    }
    row = {
        "k": k, "n_weeks": len(dates_used), "precision_at_k": series["precision"].mean(),
        "base_positive_rate": base_rate, "precision_minus_base": series["precision"].mean() - base_rate,
        "pct_weeks_at_least_one_positive": pd.Series(weekly_any_pos).mean(),
        "pct_weeks_all_positive": pd.Series(weekly_all_pos).mean(),
        "mean_selected_return": series["selected_ret"].mean(), "median_selected_return": series["selected_ret"].median(),
        "selected_minus_all_sector_mean": (series["selected_ret"] - series["all_sector_mean"]).mean(),
        "selected_minus_spy": (series["selected_ret"] - series["spy_ret"]).mean(),
        "topk_minus_bottomk_spread": (series["selected_ret"] - series["bottomk_mean"]).mean(),
        "win_rate_vs_spy": (series["selected_ret"] > series["spy_ret"]).mean(),
    }
    return row, series


# ---------------------------------------------------------------------------
# 9. Block bootstrap -- verbatim R1 method for spread-like weekly series, plus a
#    documented extension (same preserve-the-week-cross-section principle) for AUC,
#    which R1's method never needed to cover.
# ---------------------------------------------------------------------------
def block_bootstrap_ci(values: pd.Series, block_size: int = BLOCK_SIZE, n_boot: int = N_BOOTSTRAP,
                        ci: float = 0.95, seed: int = BOOTSTRAP_SEED) -> dict:
    arr = values.to_numpy()
    n = len(arr)
    rng = np.random.default_rng(seed)
    n_blocks_needed = int(ceil(n / block_size))
    max_start = n - block_size
    boot_means = np.empty(n_boot)
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        sample = np.concatenate([arr[s: s + block_size] for s in starts])[:n]
        boot_means[b] = sample.mean()
    lower, upper = np.percentile(boot_means, [2.5, 97.5])
    return {"point_estimate": arr.mean(), "ci_lower": lower, "ci_upper": upper,
            "n_weeks": n, "significant": bool(lower > 0 or upper < 0)}


def _weekly_yscore_lists(oof: pd.DataFrame):
    dates = sorted(oof["date"].unique())
    y_list, s_list = [], []
    for d in dates:
        g = oof[oof["date"] == d]
        y_list.append(g["label_binary"].to_numpy())
        s_list.append(g["predicted_proba"].to_numpy())
    return dates, y_list, s_list


def block_bootstrap_auc_edge(oof: pd.DataFrame, block_size: int = BLOCK_SIZE, n_boot: int = N_BOOTSTRAP,
                              seed: int = BOOTSTRAP_SEED) -> dict:
    """Extends R1's moving-block-bootstrap principle (resample blocks of weeks, keep each
    week's full row set intact) to AUC, which R1's own method never had to compute."""
    dates, y_list, s_list = _weekly_yscore_lists(oof)
    n = len(dates)
    rng = np.random.default_rng(seed)
    n_blocks_needed = int(ceil(n / block_size))
    max_start = n - block_size
    point_auc = roc_auc_score(oof["label_binary"], oof["predicted_proba"])
    boot_aucs = np.full(n_boot, np.nan)
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        idxs = np.concatenate([np.arange(s, s + block_size) for s in starts])[:n]
        y = np.concatenate([y_list[i] for i in idxs])
        s = np.concatenate([s_list[i] for i in idxs])
        if len(np.unique(y)) > 1:
            boot_aucs[b] = roc_auc_score(y, s)
    valid = boot_aucs[~np.isnan(boot_aucs)]
    lower, upper = np.percentile(valid, [2.5, 97.5])
    return {"point_estimate": point_auc - 0.5, "ci_lower": lower - 0.5, "ci_upper": upper - 0.5,
            "n_weeks": n, "significant": bool((lower - 0.5) > 0 or (upper - 0.5) < 0)}


def paired_block_bootstrap_auc_diff(oof_a: pd.DataFrame, oof_b: pd.DataFrame, block_size: int = BLOCK_SIZE,
                                     n_boot: int = N_BOOTSTRAP, seed: int = BOOTSTRAP_SEED) -> dict:
    """Paired version: same sampled weeks applied to both A and B each iteration, so the
    difference series inherits the correct (positively correlated) covariance structure."""
    common_dates = sorted(set(oof_a["date"].unique()) & set(oof_b["date"].unique()))
    oof_a = oof_a[oof_a["date"].isin(common_dates)]
    oof_b = oof_b[oof_b["date"].isin(common_dates)]
    dates_a, y_list_a, s_list_a = _weekly_yscore_lists(oof_a)
    dates_b, y_list_b, s_list_b = _weekly_yscore_lists(oof_b)
    assert dates_a == dates_b, "paired bootstrap requires identical week sets"
    n = len(dates_a)
    rng = np.random.default_rng(seed)
    n_blocks_needed = int(ceil(n / block_size))
    max_start = n - block_size
    point_auc_a = roc_auc_score(oof_a["label_binary"], oof_a["predicted_proba"])
    point_auc_b = roc_auc_score(oof_b["label_binary"], oof_b["predicted_proba"])
    boot_diffs = np.full(n_boot, np.nan)
    for it in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        idxs = np.concatenate([np.arange(s, s + block_size) for s in starts])[:n]
        y_a = np.concatenate([y_list_a[i] for i in idxs]); s_a = np.concatenate([s_list_a[i] for i in idxs])
        y_b = np.concatenate([y_list_b[i] for i in idxs]); s_b = np.concatenate([s_list_b[i] for i in idxs])
        if len(np.unique(y_a)) > 1 and len(np.unique(y_b)) > 1:
            boot_diffs[it] = roc_auc_score(y_a, s_a) - roc_auc_score(y_b, s_b)
    valid = boot_diffs[~np.isnan(boot_diffs)]
    lower, upper = np.percentile(valid, [2.5, 97.5])
    point = point_auc_a - point_auc_b
    return {"point_estimate": point, "ci_lower": lower, "ci_upper": upper, "n_weeks": n,
            "significant": bool(lower > 0 or upper < 0)}


print(f"[{time.time()-t0:.0f}s] Step 6/7/9/12 (evaluators + bootstrap utilities) defined.")

[3s] Step 6/7/9/12 (evaluators + bootstrap utilities) defined.


## 10. Run: raw sequence, ordered, all N x 3 models.

In [5]:
fold_metrics_rows = []
aggregate_rows = []
topk_rows = []
bootstrap_rows = []
oof_store = {}  # (N, model, "ordered") -> oof df

for N in N_VALUES:
    meta = samples_by_N[N]["meta"]
    X_seq = samples_by_N[N]["X_seq"]
    X_onehot = samples_by_N[N]["X_onehot"]
    years_N = sorted(meta["year"].unique())
    n_features = N * 11 + 11

    for model_name in ["LR", "NN_v1", "NN_v2"]:
        if model_name == "LR":
            oof, folds = walk_forward_raw_lr(meta, X_seq, X_onehot, years_N, test_years)
        elif model_name == "NN_v1":
            oof, folds = walk_forward_raw_nn(meta, X_seq, X_onehot, years_N, test_years, n_features, v2=False)
        else:
            oof, folds = walk_forward_raw_nn(meta, X_seq, X_onehot, years_N, test_years, n_features, v2=True)

        folds.insert(0, "N", N); folds.insert(1, "model", model_name); folds.insert(2, "representation", "ordered")
        fold_metrics_rows.append(folds)
        oof_store[(N, model_name, "ordered")] = oof

        auc = roc_auc_score(oof["label_binary"], oof["predicted_proba"])
        prauc = average_precision_score(oof["label_binary"], oof["predicted_proba"])
        brier = brier_score_loss(oof["label_binary"], oof["predicted_proba"])
        base_rate = oof["label_binary"].mean()
        aggregate_rows.append({
            "N": N, "model": model_name, "representation": "ordered", "n_oof_rows": len(oof),
            "pooled_auc": auc, "pooled_pr_auc": prauc, "pooled_brier": brier,
            "mean_fold_auc": folds["fold_auc"].mean(), "median_fold_auc": folds["fold_auc"].median(),
            "std_fold_auc": folds["fold_auc"].std(), "mean_fold_pr_auc": folds["fold_pr_auc"].mean(),
            "mean_fold_brier": folds["fold_brier"].mean(), "base_positive_rate": base_rate,
        })

        for k in RANK_K_VALUES:
            row, series = topk_metrics_and_series(oof, base_rate, k)
            row.update({"N": N, "model": model_name, "representation": "ordered"})
            topk_rows.append(row)

            b = block_bootstrap_ci(series["precision"])
            bootstrap_rows.append({"N": N, "model": model_name, "k": k, "quantity": "precision_minus_base",
                                    "point_estimate": b["point_estimate"] - base_rate,
                                    "ci_lower": b["ci_lower"] - base_rate, "ci_upper": b["ci_upper"] - base_rate,
                                    "n_weeks": b["n_weeks"], "significant": b["significant"]})
            for qname, diff in [
                ("selected_minus_all_sector_mean", series["selected_ret"] - series["all_sector_mean"]),
                ("selected_minus_spy", series["selected_ret"] - series["spy_ret"]),
                ("topk_minus_bottomk_spread", series["selected_ret"] - series["bottomk_mean"]),
            ]:
                b = block_bootstrap_ci(diff)
                bootstrap_rows.append({"N": N, "model": model_name, "k": k, "quantity": qname, **b})

        b = block_bootstrap_auc_edge(oof)
        bootstrap_rows.append({"N": N, "model": model_name, "k": np.nan, "quantity": "auc_edge_over_0.5", **b})

        print(f"[{time.time()-t0:.0f}s] N={N} {model_name} ordered: pooled_auc={auc:.4f} "
              f"pr_auc={prauc:.4f} brier={brier:.4f}")

print(f"[{time.time()-t0:.0f}s] Step 3/7/8/10/11/12 (ordered N x model runs) done.")

# ---------------------------------------------------------------------------
# 11. STEP 13 -- ordered vs. time-shuffled diagnostic (LR + NN v2 only, all N).
#     Fixed, documented permutation: one global permutation per N (seed=SHUFFLE_SEED),
#     applied identically to every sample's N-week window before flattening.
# ---------------------------------------------------------------------------
shuffle_rows = []
for N in N_VALUES:
    perm = np.random.default_rng(SHUFFLE_SEED).permutation(N)
    meta = samples_by_N[N]["meta"]
    X_seq = samples_by_N[N]["X_seq"]
    X_onehot = samples_by_N[N]["X_onehot"]
    X_seq_3d = X_seq.reshape(-1, N, 11)
    X_seq_shuffled = X_seq_3d[:, perm, :].reshape(-1, N * 11)
    years_N = sorted(meta["year"].unique())
    n_features = N * 11 + 11

    for model_name in ["LR", "NN_v2"]:
        if model_name == "LR":
            oof_sh, folds_sh = walk_forward_raw_lr(meta, X_seq_shuffled, X_onehot, years_N, test_years)
        else:
            oof_sh, folds_sh = walk_forward_raw_nn(meta, X_seq_shuffled, X_onehot, years_N, test_years,
                                                    n_features, v2=True)
        oof_ord = oof_store[(N, model_name, "ordered")]
        auc_ord = roc_auc_score(oof_ord["label_binary"], oof_ord["predicted_proba"])
        auc_sh = roc_auc_score(oof_sh["label_binary"], oof_sh["predicted_proba"])
        base_rate = oof_sh["label_binary"].mean()

        row_ord2, _ = topk_metrics_and_series(oof_ord, base_rate, 2)
        row_sh2, _ = topk_metrics_and_series(oof_sh, base_rate, 2)
        row_ord3, _ = topk_metrics_and_series(oof_ord, base_rate, 3)
        row_sh3, _ = topk_metrics_and_series(oof_sh, base_rate, 3)

        shuffle_rows.append({
            "N": N, "model": model_name,
            "auc_ordered": auc_ord, "auc_shuffled": auc_sh, "auc_diff_ordered_minus_shuffled": auc_ord - auc_sh,
            "precision_at_2_ordered": row_ord2["precision_at_k"], "precision_at_2_shuffled": row_sh2["precision_at_k"],
            "precision_at_3_ordered": row_ord3["precision_at_k"], "precision_at_3_shuffled": row_sh3["precision_at_k"],
            "topk_minus_bottomk_k2_ordered": row_ord2["topk_minus_bottomk_spread"],
            "topk_minus_bottomk_k2_shuffled": row_sh2["topk_minus_bottomk_spread"],
            "topk_minus_bottomk_k3_ordered": row_ord3["topk_minus_bottomk_spread"],
            "topk_minus_bottomk_k3_shuffled": row_sh3["topk_minus_bottomk_spread"],
        })
        print(f"[{time.time()-t0:.0f}s] N={N} {model_name} shuffle diagnostic: "
              f"auc_ordered={auc_ord:.4f} auc_shuffled={auc_sh:.4f}")

r3_ordered_vs_shuffled = pd.DataFrame(shuffle_rows)
r3_ordered_vs_shuffled.to_csv(OUTPUTS_ROOT / "r3_ordered_vs_shuffled.csv", index=False)
print(f"[{time.time()-t0:.0f}s] Step 13 (ordered vs shuffled) done.")

# ---------------------------------------------------------------------------
# 12. Regenerate the R1 engineered-feature baseline's OOF predictions (full 9-group
#     feature set only) for the paired comparison -- explicitly permitted by Step 9.
# ---------------------------------------------------------------------------
FEATURE_GROUPS = {
    "A": ["abs_return_1w", "abs_return_2w", "abs_return_4w", "abs_return_8w", "abs_return_12w", "abs_return_26w"],
    "B": ["excess_return_vs_spy_1w", "excess_return_vs_spy_2w", "excess_return_vs_spy_4w",
          "excess_return_vs_spy_8w", "excess_return_vs_spy_12w", "excess_return_vs_spy_26w"],
    "C": ["cross_sectional_rank_1w", "cross_sectional_rank_4w", "cross_sectional_rank_8w", "cross_sectional_rank_12w"],
    "D": ["realized_vol_4w", "realized_vol_12w", "realized_vol_26w"],
    "E": ["trend_dist_from_ma_4w", "trend_dist_from_ma_12w", "trend_dist_from_ma_26w"],
    "F": ["drawdown_dist_from_high_13w", "drawdown_dist_from_high_26w"],
    "G": ["volume_change_1w", "volume_vs_avg_4w", "volume_vs_avg_12w"],
    "H": ["spy_return_1w", "spy_return_4w", "spy_return_12w", "spy_volatility_4w"],
    "I": ["sector_return_dispersion_1w", "average_sector_return_1w", "sectors_outperforming_spy_1w"],
}
FULL_FEATURE_COLS = [c for g in "ABCDEFGHI" for c in FEATURE_GROUPS[g]]

md_r1 = panel.dropna(subset=FEATURE_COLUMNS + ["target_abs_1pct_next_week"]).copy()
md_r1["label_binary"] = md_r1["target_abs_1pct_next_week"].astype(int)
years_r1 = sorted(md_r1["year"].unique())


def walk_forward_engineered_lr(md, feature_cols, years_, test_years_):
    oof_frames = []
    for test_year in test_years_:
        train_mask = md["year"] < test_year
        test_mask = md["year"] == test_year
        scaler = StandardScaler().fit(md.loc[train_mask, feature_cols])
        X_train = scaler.transform(md.loc[train_mask, feature_cols])
        X_test = scaler.transform(md.loc[test_mask, feature_cols])
        y_train = md.loc[train_mask, "label_binary"]
        clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
        fold = md.loc[test_mask, ["date", "symbol", "next_week_open_to_close_return",
                                   "next_week_spy_return", "label_binary"]].copy()
        fold["predicted_proba"] = clf.predict_proba(X_test)[:, 1]
        oof_frames.append(fold)
    return pd.concat(oof_frames, ignore_index=True)


def walk_forward_engineered_nn(md, feature_cols, years_, test_years_, v2=False):
    oof_frames = []
    for test_year in test_years_:
        train_years_available = [y for y in years_ if y < test_year]
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]
        train_mask = md["year"].isin(train_years_available)
        fit_mask = md["year"].isin(fit_years)
        val_mask = md["year"] == val_year
        test_mask = md["year"] == test_year
        scaler = StandardScaler().fit(md.loc[train_mask, feature_cols])
        X_fit = scaler.transform(md.loc[fit_mask, feature_cols])
        X_val = scaler.transform(md.loc[val_mask, feature_cols])
        X_test = scaler.transform(md.loc[test_mask, feature_cols])
        y_fit = md.loc[fit_mask, "label_binary"].to_numpy()
        y_val = md.loc[val_mask, "label_binary"].to_numpy()
        if v2:
            seed_predictions = []
            for seed_offset in range(N_SEEDS):
                model = train_fold_mlp_v2(X_fit, y_fit, X_val, y_val, len(feature_cols), SEED_BASE + seed_offset)
                with torch.no_grad():
                    seed_predictions.append(torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy())
            predicted_proba = np.mean(seed_predictions, axis=0)
        else:
            model = train_fold_mlp(X_fit, y_fit, X_val, y_val, n_features=len(feature_cols))
            with torch.no_grad():
                predicted_proba = torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy()
        fold = md.loc[test_mask, ["date", "symbol", "next_week_open_to_close_return",
                                   "next_week_spy_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)
    return pd.concat(oof_frames, ignore_index=True)


r1_baseline_oof = {}
r1_baseline_oof["LR"] = walk_forward_engineered_lr(md_r1, FULL_FEATURE_COLS, years_r1, test_years)
r1_baseline_oof["NN_v1"] = walk_forward_engineered_nn(md_r1, FULL_FEATURE_COLS, years_r1, test_years, v2=False)
r1_baseline_oof["NN_v2"] = walk_forward_engineered_nn(md_r1, FULL_FEATURE_COLS, years_r1, test_years, v2=True)
for name, oof in r1_baseline_oof.items():
    oof.assign(model=name).to_csv(OUTPUTS_ROOT / f"r3_r1_baseline_oof_{name}.csv", index=False)
print(f"[{time.time()-t0:.0f}s] Step 9 (R1 baseline OOF regeneration for pairing) done.")

# ---------------------------------------------------------------------------
# 13. STEP 12 -- paired bootstrap comparison, R3 (best-N-per-model) vs R1 baseline.
#     "Best N per model" chosen by pooled AUC among the *ordered* runs -- report,
#     not cherry-pick: comparison run for EVERY N, not just the best.
# ---------------------------------------------------------------------------
paired_rows = []
for N in N_VALUES:
    for model_name in ["LR", "NN_v1", "NN_v2"]:
        oof_r3 = oof_store[(N, model_name, "ordered")]
        oof_r1 = r1_baseline_oof[model_name]
        auc_diff = paired_block_bootstrap_auc_diff(oof_r3, oof_r1)
        paired_rows.append({"N": N, "model": model_name, "quantity": "auc", **auc_diff})

        base_rate_r3 = oof_r3["label_binary"].mean()
        base_rate_r1 = oof_r1["label_binary"].mean()
        for k in RANK_K_VALUES:
            _, series_r3 = topk_metrics_and_series(oof_r3, base_rate_r3, k)
            _, series_r1 = topk_metrics_and_series(oof_r1, base_rate_r1, k)
            df3 = pd.DataFrame({"date": series_r3["date"], "precision": series_r3["precision"],
                                 "spread": series_r3["selected_ret"] - series_r3["bottomk_mean"],
                                 "spy_edge": series_r3["selected_ret"] - series_r3["spy_ret"]}).set_index("date")
            df1 = pd.DataFrame({"date": series_r1["date"], "precision": series_r1["precision"],
                                 "spread": series_r1["selected_ret"] - series_r1["bottomk_mean"],
                                 "spy_edge": series_r1["selected_ret"] - series_r1["spy_ret"]}).set_index("date")
            common = df3.join(df1, lsuffix="_r3", rsuffix="_r1", how="inner")
            for qname, col in [("precision_minus_base_diff", "precision"), ("topk_minus_bottomk_diff", "spread"),
                                ("topk_minus_spy_diff", "spy_edge")]:
                diff_series = common[f"{col}_r3"] - common[f"{col}_r1"]
                if qname == "precision_minus_base_diff":
                    diff_series = diff_series  # base-rate cancels in the difference of two precision series
                b = block_bootstrap_ci(diff_series)
                paired_rows.append({"N": N, "model": model_name, "k": k, "quantity": qname, **b})

r3_vs_r1_paired = pd.DataFrame(paired_rows)
r3_vs_r1_paired.to_csv(OUTPUTS_ROOT / "r3_vs_r1_paired_comparison.csv", index=False)
print(f"[{time.time()-t0:.0f}s] Step 12 (paired bootstrap vs R1 baseline) done.")

[16s] N=4 LR ordered: pooled_auc=0.5313 pr_auc=0.3818 brier=0.2557


[30s] N=4 NN_v1 ordered: pooled_auc=0.5446 pr_auc=0.3748 brier=0.2328


[47s] N=4 NN_v2 ordered: pooled_auc=0.5369 pr_auc=0.3728 brier=0.2298


[61s] N=8 LR ordered: pooled_auc=0.5304 pr_auc=0.3811 brier=0.2680


[74s] N=8 NN_v1 ordered: pooled_auc=0.5284 pr_auc=0.3691 brier=0.2292


...

[328s] N=26 NN_v2 shuffle diagnostic: auc_ordered=0.5274 auc_shuffled=0.5328


[356s] N=52 LR shuffle diagnostic: auc_ordered=0.5225 auc_shuffled=0.5226


[371s] N=52 NN_v2 shuffle diagnostic: auc_ordered=0.5358 auc_shuffled=0.5346
[371s] Step 13 (ordered vs shuffled) done.


[391s] Step 9 (R1 baseline OOF regeneration for pairing) done.


[747s] Step 12 (paired bootstrap vs R1 baseline) done.


## 14. Save the remaining required artifacts.

In [6]:
r3_fold_metrics = pd.concat(fold_metrics_rows, ignore_index=True)
r3_fold_metrics.to_csv(OUTPUTS_ROOT / "r3_fold_metrics.csv", index=False)

r3_aggregate_metrics = pd.DataFrame(aggregate_rows)
r3_aggregate_metrics.to_csv(OUTPUTS_ROOT / "r3_aggregate_metrics.csv", index=False)

r3_topk_all = pd.DataFrame(topk_rows)
r3_topk_all[r3_topk_all["k"] == 2].to_csv(OUTPUTS_ROOT / "r3_top2_metrics.csv", index=False)
r3_topk_all[r3_topk_all["k"] == 3].to_csv(OUTPUTS_ROOT / "r3_top3_metrics.csv", index=False)

r3_bootstrap_ci = pd.DataFrame(bootstrap_rows)
r3_bootstrap_ci.to_csv(OUTPUTS_ROOT / "r3_bootstrap_ci.csv", index=False)

# R1 engineered baseline metrics, for direct citation in the report (exact artifact values).
r1_ablation_full = pd.read_csv(OUTPUTS_ROOT / "r1_primary_lr_ablation_walk_forward_results.csv")
r1_nn1_full = pd.read_csv(OUTPUTS_ROOT / "r1_primary_nn_v1_ablation_walk_forward_results.csv")
r1_nn2_full = pd.read_csv(OUTPUTS_ROOT / "r1_primary_nn_v2_ablation_walk_forward_results.csv")
r1_baselines = {
    "LR_best_of_9_sets_auc": float(r1_ablation_full["auc"].max()),
    "NN_v1_best_of_9_sets_auc": float(r1_nn1_full["auc"].max()),
    "NN_v2_best_of_9_sets_auc": float(r1_nn2_full["auc"].max()),
    "LR_full_feature_set_auc": float(r1_ablation_full[r1_ablation_full["feature_set"] == "A+B+C+D+E+F+G+H+I"]["auc"].iloc[0]),
    "NN_v1_full_feature_set_auc": float(r1_nn1_full[r1_nn1_full["feature_set"] == "A+B+C+D+E+F+G+H+I"]["auc"].iloc[0]),
    "NN_v2_full_feature_set_auc": float(r1_nn2_full[r1_nn2_full["feature_set"] == "A+B+C+D+E+F+G+H+I"]["auc"].iloc[0]),
}

best_ordered = r3_aggregate_metrics[r3_aggregate_metrics["representation"] == "ordered"]
summary = {
    "n_values_tested": N_VALUES,
    "r3_best_lr_auc": float(best_ordered[best_ordered.model == "LR"]["pooled_auc"].max()),
    "r3_best_nn_v1_auc": float(best_ordered[best_ordered.model == "NN_v1"]["pooled_auc"].max()),
    "r3_best_nn_v2_auc": float(best_ordered[best_ordered.model == "NN_v2"]["pooled_auc"].max()),
    "r1_engineered_baselines": r1_baselines,
    "vde_gap_sequence_samples_invalidated": 0,
    "runtime_seconds": time.time() - t0,
}
with open(OUTPUTS_ROOT / "r3_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"[{time.time()-t0:.0f}s] ALL DONE.")
print(json.dumps(summary, indent=2))

[747s] ALL DONE.
{
  "n_values_tested": [
    4,
    8,
    13,
    26,
    52
  ],
  "r3_best_lr_auc": 0.5313449419173871,
  "r3_best_nn_v1_auc": 0.5446183031927729,
  "r3_best_nn_v2_auc": 0.5368587457337194,
  "r1_engineered_baselines": {
    "LR_best_of_9_sets_auc": 0.5615520548476269,
    "NN_v1_best_of_9_sets_auc": 0.5721381689562363,
    "NN_v2_best_of_9_sets_auc": 0.5750523190924745,
    "LR_full_feature_set_auc": 0.5460421501553522,
    "NN_v1_full_feature_set_auc": 0.5680100994943317,
    "NN_v2_full_feature_set_auc": 0.5735730758971164
  },
  "vde_gap_sequence_samples_invalidated": 0,
  "runtime_seconds": 746.7676868438721
}
